In [7]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras import layers, Model, callbacks, optimizers
from tensorflow.keras import utils
import matplotlib.pyplot as plt
import ipywidgets as widgets


In [8]:
# read data
df_data = pd.read_parquet('../../Data/training_data.parquet.gzip')
print(df_data.shape)
len(df_data.index.get_level_values(0).unique())

(1825488, 8)


336

In [9]:
# calculate daily changes by ticker
df_analysis = df_data.groupby(level='ticker', as_index=False).apply(pd.DataFrame.pct_change)

# calculate the standard deviation of close price daily change by ticker
df_analysis = df_analysis.groupby(level='ticker', as_index=True)['Close'].apply(np.std)

# sort valeus
df_analysis.sort_values(inplace=True)

# pick tickers: 2*most_volatile, 2*most_stable, 2*in_middle
mid = int(len(df_analysis.index)/2)
tickers_ft = df_analysis.index[:5].append(df_analysis.index[mid:mid+5]).append(df_analysis.index[-5:])
tickers_ft = tickers_ft.tolist()[:10]
tickers_ft

['CET', 'AJG', 'FNWD', 'CFFN', 'RY', 'FDBC', 'VABK', 'FCF', 'PFC', 'BUSE']

In [10]:
# common variables

seq_length = 7   # use the last 7 days to predict the next day
features = ["Open", "High", "Low", "Close"]
seq_length = 7
idx = pd.IndexSlice


In [11]:
def get_ticker_df(ticker, features=features):
    return df_data.loc[idx[ticker, :], features]

def create_sequence_dataset(df, seq_length=seq_length, batch_size=None, shuffle=False, seed=88):
    return utils.timeseries_dataset_from_array(
        data=df.to_numpy(dtype=np.float32),
        targets=df["Close"].iloc[seq_length:].to_numpy(dtype=np.float32),
        sequence_length=seq_length,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
    )

def dataset_to_numpy(dataset):
    x_list, y_list = [], []
    for x, y in dataset:
        x_list.append(x.numpy())
        y_list.append(y.numpy())
    return np.stack(x_list), np.asarray(y_list)

## Similarity, Graph, Evaluate by dataset

In [12]:
import sys
import os
sys.path.insert(0, '.')
from CKA import linear_CKA

model_layers = [9]
model_states = ['optimal']

os.makedirs('retrain_outputs_CKA', exist_ok=True)

evaluation_mape = []

for num_layers in model_layers:
    for state in model_states:
        MODEL_PATH = f'retrain_saved_models/Retrained_{num_layers}_Layers_{state}.keras'
        model = tf.keras.models.load_model(MODEL_PATH)
        model.compile(loss="mean_absolute_percentage_error")

        # Build a model returning all representations except the final Dense output:
        # all_reprs[0] = model input, all_reprs[k] = k-th hidden LSTM output
        all_layer_outputs = [model.layers[i].output for i in range(len(model.layers) - 1)]
        repr_model = tf.keras.Model(inputs=model.input, outputs=all_layer_outputs)

        for ticker in tickers_ft:
            df_ticker = get_ticker_df(ticker)
            dataset = create_sequence_dataset(df_ticker, batch_size=None, shuffle=False)
            x_all, y_all = dataset_to_numpy(dataset)

            # all_reprs[j]: shape (n_samples, seq_len, d_j)
            all_reprs = repr_model.predict(x_all, verbose=0)

            n_samples = x_all.shape[0]
            n_hidden = len(model.layers) - 2  # exclude InputLayer and Dense

            # Per-sample intra-layer CKA: for each sample, compute CKA(layer_input, layer_output)
            # treating the seq_len timesteps as the sample dimension within each data sample
            cka_matrix = np.zeros((n_samples, 1))
            for sample_idx in range(n_samples):
                for layer_idx in range(n_hidden):
                    x_in  = all_reprs[0][sample_idx]         # (seq_len, d_in)
                    x_out = all_reprs[n_hidden][sample_idx]  # (seq_len, d_out)
                    cka_matrix[sample_idx, 0] = linear_CKA(x_in, x_out)

            # Save CKA results as CSV
            df_cka = pd.DataFrame(cka_matrix, columns=["CKA"])
            df_cka.index.name = "sample_idx"
            df_cka.to_csv(f'retrain_outputs_CKA/whole_CKA/{num_layers}Layers_{state}_{ticker}_cka.csv')



2026-06-19 20:29:51.776240: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 20:29:55.209810: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 20:29:58.134869: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/home/mchen/PraxisDoc/Experiments/FIN_experiment_cka/./CKA.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return hsic / (var1 * var2)
2026-06-19 20:30:01.111839: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 20:30:04.188400: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 20:30:07.281131: W tensorflow/core/framework/local_rendezvous.cc:404]